<a href="https://colab.research.google.com/github/milk1833/4IZ503-movielens-imdb/blob/main/01_priprava_dat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 – Příprava dat: MovieLens + IMDb

**Projekt 4IZ503 – Fáze 1 (Extract + Transform)**

Co tento notebook dělá:

1. **Stáhne** data MovieLens a IMDb (přímo z internetu → projekt je reprodukovatelný).
2. **Načte** je a IMDb ořízne jen na filmy, které jsou v MovieLens (IMDb má miliony řádků).
3. **Spojí** oba zdroje přes klíč `imdbId` ↔ `tconst`.
4. **Vyčistí** data (chybějící hodnoty, typy, rok z názvu, datum z timestampu).
5. **Spočítá odvozené ukazatele** (rozdíl MovieLens vs. IMDb, dekáda, kategorie délky…).
6. **Rozdělí** data do tabulek hvězdicového schématu (fakta + dimenze) pro PostgreSQL.
7. **Ověří správnost** (počty řádků, spárovanost, referenční integrita).
8. **Uloží** výsledné CSV do `data/processed/`.

Spouštějte buňky **odshora dolů** (v Colabu: *Runtime → Run all*).

## 1. Nastavení

Používáme **`ml-25m`** – stejnou verzi jako ve schválené úvodní zprávě
(25 mil. hodnocení, ~62 tis. filmů, ~162 tis. uživatelů).

**Vzorek uživatelů (`USER_SAMPLE_FRAC`).** Všech 25 mil. hodnocení by se špatně vešlo do Colabu, GitHubu
i Power BI. Proto:
- **ukazatele za film** (průměr, počet hodnocení, podíl vysokých hodnocení, počet tagů) počítáme z **celých dat**,
- do **faktové tabulky** hodnocení dáme jen **náhodný vzorek uživatelů** (výchozí 10 %, ≈ 2,5 mil. řádků).

Vzorkujeme **uživatele, ne jednotlivá hodnocení** – každý vybraný uživatel má celou svou historii,
takže profily uživatelů (aktivita, přísnost) zůstávají věrné. `SEED` zajišťuje, že vzorek je při každém
spuštění stejný (reprodukovatelnost). Toto rozhodnutí umějte u zkoušky zdůvodnit.

**Zmrazení dat na Google Disku (`USE_DRIVE = True`).** IMDb se aktualizuje každý den a starší verze nearchivuje.
Proto se surová data ukládají na váš Google Disk do `MyDrive/4IZ503/data/raw`. Při prvním spuštění se stáhnou,
při každém dalším se použijí **tytéž soubory** → čísla se během práce nemění. Datum stažení se zapíše
do souboru `_datum_stazeni.txt` (uveďte ho v README a v prezentaci).
Colab si při prvním spuštění vyžádá povolení přístupu k Disku – potvrďte ho.

In [1]:
import os, zipfile, urllib.request, csv
import numpy as np
import pandas as pd

ML_VERSION = "ml-25m"         # stejná verze jako v úvodní zprávě (pro rychlý test: "ml-latest-small")
USER_SAMPLE_FRAC = 0.10       # podíl uživatelů ve faktové tabulce (1.0 = všichni)
SEED = 42                     # pevné "semínko" náhody -> stejný vzorek při každém spuštění
USE_DRIVE = True              # True = surová data i výstupy na Google Disku (zmrazená verze)

RAW, OUT = "data/raw", "data/processed"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = "/content/drive/MyDrive/4IZ503"
        RAW, OUT = f"{BASE}/data/raw", f"{BASE}/data/processed"
    except ImportError:
        print("Nejste v Colabu – používám lokální složky data/raw a data/processed")
os.makedirs(RAW, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)

Mounted at /content/drive
pandas 2.2.3


## 2. Stažení dat

Funkce `download` stáhne soubor jen tehdy, když už na disku není (při opakovaném spuštění se nestahuje znovu).

- MovieLens: https://grouplens.org/datasets/movielens/
- IMDb: https://datasets.imdbws.com/ (popis souborů: https://developer.imdb.com/non-commercial-datasets/)

In [2]:
def download(url, path):
    if os.path.exists(path):
        print("už staženo:", path)
    else:
        print("stahuji:", url)
        urllib.request.urlretrieve(url, path)
    return path

# MovieLens (zip) -> rozbalit
ml_zip = download(f"https://files.grouplens.org/datasets/movielens/{ML_VERSION}.zip",
                  f"{RAW}/{ML_VERSION}.zip")
ML_DIR = f"{RAW}/{ML_VERSION}"
if not os.path.exists(ML_DIR):
    with zipfile.ZipFile(ml_zip) as z:
        z.extractall(RAW)

# IMDb (gzipované TSV) – nerozbalujeme, pandas umí číst .gz přímo
IMDB_FILES = ["title.basics", "title.ratings", "title.crew", "name.basics"]
from datetime import date
DATE_FILE = f"{RAW}/_datum_stazeni.txt"
if not all(os.path.exists(f"{RAW}/{f}.tsv.gz") for f in IMDB_FILES):
    with open(DATE_FILE, "w") as fh:          # zapíše se jen při skutečném stažení
        fh.write(str(date.today()))
for f in IMDB_FILES:
    download(f"https://datasets.imdbws.com/{f}.tsv.gz", f"{RAW}/{f}.tsv.gz")
IMDB_DATE = open(DATE_FILE).read().strip() if os.path.exists(DATE_FILE) else "neznámé"
print("\nIMDb data ze dne:", IMDB_DATE)

stahuji: https://files.grouplens.org/datasets/movielens/ml-25m.zip
stahuji: https://datasets.imdbws.com/title.basics.tsv.gz
stahuji: https://datasets.imdbws.com/title.ratings.tsv.gz
stahuji: https://datasets.imdbws.com/title.crew.tsv.gz
stahuji: https://datasets.imdbws.com/name.basics.tsv.gz

IMDb data ze dne: 2026-11-07


## 3. Načtení MovieLens

| soubor | jeden řádek = | klíč |
|---|---|---|
| `movies.csv` | film | `movieId` |
| `ratings.csv` | jedno hodnocení uživatele | `userId` + `movieId` |
| `tags.csv` | jeden štítek (tag) od uživatele | `userId` + `movieId` |
| `links.csv` | převodník ID filmu na IMDb a TMDB | `movieId` → `imdbId` |

In [3]:
movies  = pd.read_csv(f"{ML_DIR}/movies.csv")
ratings = pd.read_csv(f"{ML_DIR}/ratings.csv",     # menší datové typy -> polovina paměti
                      dtype={"userId":"int32","movieId":"int32","rating":"float32","timestamp":"int64"})
tags    = pd.read_csv(f"{ML_DIR}/tags.csv")
links   = pd.read_csv(f"{ML_DIR}/links.csv", dtype={"imdbId": str})  # text -> zachová úvodní nuly

# Zdrojové počty si uložíme – na konci je porovnáme s výsledkem
src_counts = {"movies": len(movies), "ratings": len(ratings),
              "tags": len(tags), "links": len(links)}
src_counts

{'movies': 62423, 'ratings': 25000095, 'tags': 1093360, 'links': 62423}

In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


## 4. Spojovací klíč

V `links.csv` je IMDb ID jako číslo bez prefixu (`114709`), v IMDb jako text `tt0114709`.
Převod: doplnit nuly zleva na 7 číslic a přidat `tt`. Novější filmy mají 8 číslic – `zfill(7)` je nezkrátí.

In [5]:
links["tconst"] = "tt" + links["imdbId"].str.zfill(7)
keep_tconst = set(links["tconst"])     # množina filmů, které nás z IMDb zajímají
links.head()

,movieId,imdbId,tmdbId,tconst
0,1,0114709,862.0,tt0114709
1,2,0113497,8844.0,tt0113497
2,3,0113228,15602.0,tt0113228
3,4,0114885,31357.0,tt0114885
4,5,0113041,11862.0,tt0113041


## 5. Načtení IMDb – po částech a s filtrem

IMDb soubory mají miliony řádků. Čteme je **po kouscích (chunk)** a z každého kousku necháme jen řádky,
které potřebujeme. Tím se nezaplní paměť.

Důležité parametry:
- `na_values="\\N"` – IMDb označuje chybějící hodnotu textem `\N` → převedeme na skutečnou prázdnou hodnotu,
- `quoting=csv.QUOTE_NONE` – některé názvy obsahují uvozovky, které by jinak rozbily načítání,
- `dtype=str` – vše načteme jako text, typy převedeme až ručně.

In [6]:
def read_imdb_filtered(name, key, keep, usecols=None, chunksize=500_000):
    parts = []
    reader = pd.read_csv(f"{RAW}/{name}.tsv.gz", sep="\t", na_values="\\N",
                         quoting=csv.QUOTE_NONE, dtype=str, usecols=usecols,
                         chunksize=chunksize)
    total = 0
    for chunk in reader:
        total += len(chunk)
        parts.append(chunk[chunk[key].isin(keep)])
    df = pd.concat(parts, ignore_index=True)
    print(f"{name}: přečteno {total:,} řádků, ponecháno {len(df):,}")
    return df

basics = read_imdb_filtered("title.basics", "tconst", keep_tconst,
            usecols=["tconst","titleType","primaryTitle","isAdult","startYear","runtimeMinutes","genres"])
imdb_r = read_imdb_filtered("title.ratings", "tconst", keep_tconst)
crew   = read_imdb_filtered("title.crew", "tconst", keep_tconst, usecols=["tconst","directors","writers"])

title.basics: přečteno 12,942,367 řádků, ponecháno 62,294
title.ratings: přečteno 1,729,742 řádků, ponecháno 62,240
title.crew: přečteno 12,942,367 řádků, ponecháno 62,294


### Režiséři

V `title.crew` je ve sloupci `directors` seznam ID oddělený čárkou (`nm0005124,nm0000001`).
„Rozbalíme“ ho na samostatné řádky (`explode`) – vznikne vazba **film ↔ režisér** (M:N).
Pak dohledáme jména v `name.basics` (opět s filtrem).

In [7]:
movie_dir = (crew.dropna(subset=["directors"])
                 .assign(nconst=lambda d: d["directors"].str.split(","))
                 .explode("nconst")[["tconst","nconst"]]
                 .drop_duplicates())

persons = read_imdb_filtered("name.basics", "nconst", set(movie_dir["nconst"]),
            usecols=["nconst","primaryName","birthYear","deathYear","primaryProfession"])
print("vazeb film–režisér:", len(movie_dir), "| režisérů:", movie_dir["nconst"].nunique())

name.basics: přečteno 15,795,204 řádků, ponecháno 28,277
vazeb film–režisér: 70549 | režisérů: 28277


## 6. Čištění

**MovieLens:**
- rok vydání je schovaný v názvu „Toy Story (1995)“ → vytáhneme regulárním výrazem,
- žánr „(no genres listed)“ = žádný žánr,
- `timestamp` (sekundy od 1. 1. 1970) → datum.

**IMDb:** převod textů na čísla (`errors="coerce"` = co nejde převést, bude prázdné).

In [8]:
# --- MovieLens: filmy
movies["ml_year"]  = pd.to_numeric(movies["title"].str.extract(r"\((\d{4})\)\s*$")[0], errors="coerce")
movies["title_clean"] = movies["title"].str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True).str.strip()
movies["genres"] = movies["genres"].replace("(no genres listed)", np.nan)

# --- MovieLens: tagy (u hodnocení převedeme datum až po výběru vzorku – šetří paměť)
tags["tagged_at"] = pd.to_datetime(tags["timestamp"], unit="s")
tags["tag"] = tags["tag"].astype(str).str.strip().str.lower()   # "Pixar" a "pixar" = stejný tag

# --- IMDb: typy
for c in ["startYear","runtimeMinutes"]:
    basics[c] = pd.to_numeric(basics[c], errors="coerce")
basics = basics.rename(columns={"genres": "imdb_genres"})   # aby se nepletl se žánry z MovieLens
basics["isAdult"] = pd.to_numeric(basics["isAdult"], errors="coerce").fillna(0).astype(int)

# Stejně jako v úvodní zprávě: tituly pro dospělé (isAdult = 1) z IMDb vyřazujeme
adult_ids = set(basics.loc[basics["isAdult"] == 1, "tconst"])
basics = basics[basics["isAdult"] == 0].drop(columns="isAdult")
print("vyřazeno adult titulů:", len(adult_ids))
imdb_r["averageRating"] = pd.to_numeric(imdb_r["averageRating"], errors="coerce")
imdb_r["numVotes"]      = pd.to_numeric(imdb_r["numVotes"], errors="coerce")
for c in ["birthYear","deathYear"]:
    persons[c] = pd.to_numeric(persons[c], errors="coerce")

# Kontrola duplicit klíčů (musí být 0, jinak by spojení "rozmnožilo" řádky)
print("duplicitní movieId:", movies["movieId"].duplicated().sum())
print("duplicitní tconst v links:", links["tconst"].duplicated().sum())
print("duplicitní tconst v IMDb basics:", basics["tconst"].duplicated().sum())

vyřazeno adult titulů: 29
duplicitní movieId: 0
duplicitní tconst v links: 0
duplicitní tconst v IMDb basics: 0


## 7. Spojení MovieLens + IMDb (úroveň filmu)

Používáme **LEFT JOIN** od MovieLens: každý film z MovieLens zůstane, i když ho IMDb nezná
(pak má IMDb sloupce prázdné). Díky tomu žádný film „nezmizí“ a můžeme spočítat, kolik se jich nespárovalo.

`validate="one_to_one"` / `"many_to_one"` – pandas zkontroluje, že spojení je opravdu 1:1, jinak skončí chybou.

In [9]:
movie = (movies
         .merge(links[["movieId","imdbId","tmdbId","tconst"]], on="movieId", how="left", validate="one_to_one")
         .merge(basics, on="tconst", how="left", validate="many_to_one")
         .merge(imdb_r, on="tconst", how="left", validate="many_to_one"))

assert len(movie) == src_counts["movies"], "spojení změnilo počet filmů!"

matched = movie["titleType"].notna()
print(f"Filmů v MovieLens: {len(movie):,}")
print(f"Spárováno s IMDb:  {matched.sum():,} ({matched.mean():.1%})")
print(f"S IMDb hodnocením: {movie['averageRating'].notna().sum():,}")
print(f"(Úvodní zpráva, IMDb ze 17. 10. 2026: 158 nespárovaných, z toho 29 adult."
      f" Teď IMDb ze dne {IMDB_DATE} – drobný rozdíl je dán aktualizací IMDb.)")
print(f"Nespárováno kvůli vyřazeným adult titulům: {movie.loc[~matched, 'tconst'].isin(adult_ids).sum():,}")
print("\nTypy titulů v IMDb:")
print(movie["titleType"].value_counts(dropna=False))

Filmů v MovieLens: 62,423
Spárováno s IMDb:  62,265 (99.7%)
S IMDb hodnocením: 62,240
(Úvodní zpráva, IMDb ze 17. 10. 2026: 158 nespárovaných, z toho 29 adult. Teď IMDb ze dne 2026-11-07 – drobný rozdíl je dán aktualizací IMDb.)
Nespárováno kvůli vyřazeným adult titulům: 29

Typy titulů v IMDb:
titleType
movie           52814
tvMovie          3598
short            2790
video            1537
tvSpecial         527
tvMiniSeries      455
tvEpisode         369
NaN               158
tvShort            90
tvSeries           85
Name: count, dtype: int64


In [10]:
# Nespárované filmy – podívejte se na ně a důvod popište v prezentaci
movie.loc[~matched, ["movieId","title","imdbId","tconst"]].head(10)

,movieId,title,imdbId,tconst
705,720,Wallace & Gromit: The Best of Aardman Animatio...,0118114,tt0118114
2167,2258,"Master, The (1984)",0087690,tt0087690
2419,2510,Just the Ticket (1999),0134948,tt0134948
2759,2851,Saturn 3 (1980),0081454,tt0081454
2907,2999,Man of the Century (1999),0154827,tt0154827
3947,4051,Horrors of Spider Island (Ein Toter Hing im Ne...,0056600,tt0056600
4137,4241,Pokémon 3: The Movie (2001),0266860,tt0266860
5156,5264,Clockstoppers (2002),0250305,tt0250305
5295,5403,"Sleepy Time Gal, The (2001)",0277322,tt0277322
5561,5672,Pokemon 4 Ever (a.k.a. Pokémon 4: The Movie) (...,0313487,tt0313487


## 8. Odvozené ukazatele

Tady vznikají „netriviální ukazatele“, které se hodnotí. Každý umějte vysvětlit:

| ukazatel | výpočet | k čemu |
|---|---|---|
| `ml_avg_rating`, `ml_num_ratings` | průměr / počet hodnocení v MovieLens | základní míra kvality a popularity |
| `ml_share_high` | podíl hodnocení ≥ 4 | „jak často film nadchne“ (méně citlivé na extrémy než průměr) |
| `ml_avg_rating_10` | `ml_avg_rating × 2` | MovieLens je 0,5–5, IMDb 1–10 → stejná škála |
| `rating_diff` | `imdb_rating − ml_avg_rating_10` | **stejně jako v úvodní zprávě**: kladné = IMDb hodnotí lépe než MovieLens |
| `spolehlivy` | ≥ 10 hodnocení v ML **a** ≥ 100 hlasů v IMDb | filtr proti filmům s pár hodnoceními (náhodné extrémy) |
| `year` | rok z IMDb, jinak z názvu MovieLens | sjednocený rok vydání |
| `decade` | rok zaokrouhlený dolů na desítky | analýza trendů |
| `runtime_cat` | krátký ≤80 / standardní 80–100 / delší 100–130 / dlouhý >130 min (jako v úvodní zprávě) | kategorie pro BI i GUHA |
| `genre_count` | počet žánrů filmu | „čistý“ vs. žánrový mix |
| `n_directors`, `n_writers`, `n_tags` | počty | složitost produkce, „zajímavost“ pro komunitu |
| `movie_age_at_rating` | rok hodnocení − rok vydání | hodnotí lidé novinky, nebo klasiku? |

In [11]:
# --- agregace hodnocení na film
agg = ratings.groupby("movieId").agg(ml_avg_rating=("rating","mean"),
                                      ml_num_ratings=("rating","size"),
                                      ml_rating_std=("rating","std")).reset_index()
# podíl hodnocení >= 4 (vektorově, rychlé i pro 25 mil. řádků)
agg = agg.merge((ratings["rating"] >= 4).groupby(ratings["movieId"]).mean()
                    .rename("ml_share_high").reset_index(), on="movieId")
movie = movie.merge(agg, on="movieId", how="left", validate="one_to_one")
movie["ml_num_ratings"] = movie["ml_num_ratings"].fillna(0).astype(int)

# --- srovnání škál
movie["ml_avg_rating_10"]    = movie["ml_avg_rating"] * 2
movie["rating_diff"]      = movie["averageRating"] - movie["ml_avg_rating_10"]   # IMDb − MovieLens
movie["spolehlivy"] = ((movie["ml_num_ratings"] >= 10) & (movie["numVotes"] >= 100)).astype(int)

# --- rok, dekáda, délka
movie["year"]   = movie["startYear"].fillna(movie["ml_year"])
movie["decade"] = (movie["year"] // 10 * 10).astype("Int64")
# stejné hranice jako v úvodní zprávě
movie["runtime_cat"] = pd.cut(movie["runtimeMinutes"], bins=[0, 80, 100, 130, 10_000],
                              labels=["krátký", "standardní", "delší", "dlouhý"])

# --- počty
movie["genre_count"] = movie["genres"].str.split("|").str.len().fillna(0).astype(int)
movie = movie.merge(movie_dir.groupby("tconst").size().rename("n_directors").reset_index(),
                    on="tconst", how="left")
movie = movie.merge(tags.groupby("movieId").size().rename("n_tags").reset_index(),
                    on="movieId", how="left")
w = crew["writers"].fillna("").astype(str)          # "nm1,nm2" -> 2, prázdné -> 0
crew["n_writers"] = np.where(w == "", 0, w.str.count(",") + 1)
movie = movie.merge(crew[["tconst","n_writers"]],
                    on="tconst", how="left", validate="many_to_one")
movie[["n_directors","n_writers","n_tags"]] = movie[["n_directors","n_writers","n_tags"]].fillna(0).astype(int)

movie[["title_clean","year","ml_avg_rating_10","averageRating","rating_diff","spolehlivy"]].head()

,title_clean,year,ml_avg_rating_10,averageRating,rating_diff,spolehlivy
0,Toy Story,1995.0,7.787416,8.3,0.512584,1
1,Jumanji,1995.0,6.503054,7.1,0.596946,1
2,Grumpier Old Men,1995.0,6.284056,6.7,0.415944,1
3,Waiting to Exhale,1995.0,5.707095,6.1,0.392905,1
4,Father of the Bride Part II,1995.0,6.116868,6.1,-0.016868,1


## 9. Tabulky hvězdicového schématu

```
                 dim_date
                    │
dim_user ──── fact_rating ──── dim_movie ──── bridge_movie_genre ──── dim_genre
                                   │
                 fact_tag ─────────┤
                                   └── bridge_movie_director ──── dim_person
```

- **fakta** = události, které měříme (hodnocení, tagy),
- **dimenze** = popis, podle kterého filtrujeme (film, uživatel, datum, žánr, režisér),
- **bridge** = mezitabulka pro vazbu M:N (film má víc žánrů, žánr má víc filmů) – to je ta **normalizace**:
  žánry nejsou jako text „Action|Comedy“ v jedné buňce.

In [12]:
# ---------- dim_movie
dim_movie = movie.rename(columns={
    "movieId":"movie_id", "title_clean":"title", "primaryTitle":"imdb_title",
    "titleType":"imdb_title_type", "isAdult":"is_adult", "runtimeMinutes":"runtime_min",
    "averageRating":"imdb_rating", "numVotes":"imdb_num_votes", "tmdbId":"tmdb_id", "imdbId":"imdb_id"
})[["movie_id","tconst","imdb_id","tmdb_id","title","genres","imdb_genres","imdb_title","imdb_title_type",
    "year","decade","runtime_min","runtime_cat","genre_count","n_directors","n_writers","n_tags",
    "imdb_rating","imdb_num_votes","ml_avg_rating","ml_avg_rating_10","ml_num_ratings",
    "ml_rating_std","ml_share_high","rating_diff","spolehlivy"]]
dim_movie["imdb_matched"] = dim_movie["imdb_title_type"].notna().astype(int)

# ---------- dim_genre + bridge_movie_genre (žánry z MovieLens)
mg = (movies[["movieId","genres"]].dropna()
        .assign(genre=lambda d: d["genres"].str.split("|")).explode("genre"))
dim_genre = (pd.DataFrame({"genre_name": sorted(mg["genre"].unique())})
               .reset_index().rename(columns={"index":"genre_id"}))
dim_genre["genre_id"] += 1
bridge_movie_genre = (mg.merge(dim_genre, left_on="genre", right_on="genre_name")
                        [["movieId","genre_id"]].rename(columns={"movieId":"movie_id"})
                        .drop_duplicates())

# ---------- dim_person + bridge_movie_director
dim_person = persons.rename(columns={"nconst":"person_id","primaryName":"name",
                                     "birthYear":"birth_year","deathYear":"death_year",
                                     "primaryProfession":"professions"})
bridge_movie_director = (movie_dir.merge(links[["movieId","tconst"]], on="tconst")
                           [["movieId","nconst"]]
                           .rename(columns={"movieId":"movie_id","nconst":"person_id"}))
# jen režiséři, které známe v dim_person (referenční integrita)
bridge_movie_director = bridge_movie_director[bridge_movie_director["person_id"].isin(dim_person["person_id"])]

print(len(dim_movie), len(dim_genre), len(bridge_movie_genre), len(dim_person), len(bridge_movie_director))

62423 19 107245 28277 70549


### Vzorek uživatelů pro faktové tabulky

Ukazatele za film už máme spočítané z celých dat. Teď vybereme vzorek uživatelů pro `fact_rating`.
Tagy (`fact_tag`) omezíme na stejné uživatele, aby každý záznam měl svého uživatele v `dim_user`.

In [13]:
users_all = ratings["userId"].drop_duplicates()
users_s   = users_all.sample(frac=USER_SAMPLE_FRAC, random_state=SEED)
ratings_s = ratings[ratings["userId"].isin(users_s)].copy()
tags_s    = tags[tags["userId"].isin(users_s)].copy()
ratings_s["rated_at"] = pd.to_datetime(ratings_s["timestamp"], unit="s")

src_counts["ratings_sample"] = len(ratings_s)
src_counts["tags_sample"]    = len(tags_s)
print(f"Uživatelů celkem {len(users_all):,}, ve vzorku {len(users_s):,}")
print(f"Hodnocení celkem {len(ratings):,}, ve vzorku {len(ratings_s):,} ({len(ratings_s)/len(ratings):.1%})")
print(f"Průměrné hodnocení: celá data {ratings['rating'].mean():.3f} | vzorek {ratings_s['rating'].mean():.3f}"
      "  <- mají být téměř stejné (vzorek je reprezentativní)")
del ratings   # uvolnit paměť

Uživatelů celkem 162,541, ve vzorku 16,254
Hodnocení celkem 25,000,095, ve vzorku 2,503,205 (10.0%)
Průměrné hodnocení: celá data 3.534 | vzorek 3.547  <- mají být téměř stejné (vzorek je reprezentativní)


In [14]:
# ---------- dim_date (každý den, kdy proběhlo hodnocení nebo tag)
def to_date_id(ts):   # datum -> číslo 20051231 (rychlejší než převod přes text)
    return (ts.dt.year * 10000 + ts.dt.month * 100 + ts.dt.day).astype("int32")

all_dates = pd.concat([ratings_s["rated_at"], tags_s["tagged_at"]]).dt.normalize().drop_duplicates()
dim_date = pd.DataFrame({"date": all_dates.sort_values().values})
dim_date["date_id"]   = to_date_id(dim_date["date"])
dim_date["year"]      = dim_date["date"].dt.year
dim_date["quarter"]   = dim_date["date"].dt.quarter
dim_date["month"]     = dim_date["date"].dt.month
dim_date["weekday"]   = dim_date["date"].dt.dayofweek + 1                     # 1 = pondělí
dim_date["is_weekend"]= (dim_date["weekday"] >= 6).astype(int)
dim_date = dim_date[["date_id","date","year","quarter","month","weekday","is_weekend"]]

# ---------- fact_rating
fact_rating = ratings_s.rename(columns={"userId":"user_id","movieId":"movie_id"}).copy()
fact_rating.insert(0, "rating_id", range(1, len(fact_rating) + 1))
fact_rating["date_id"] = to_date_id(fact_rating["rated_at"])
fact_rating = fact_rating.merge(dim_movie[["movie_id","year"]], on="movie_id", how="left", validate="many_to_one")
fact_rating["movie_age_at_rating"] = (fact_rating["rated_at"].dt.year - fact_rating["year"]).astype("Int64")
fact_rating = fact_rating[["rating_id","user_id","movie_id","date_id","rating","rated_at","movie_age_at_rating"]]

# ---------- fact_tag
fact_tag = tags_s.rename(columns={"userId":"user_id","movieId":"movie_id"}).copy()
fact_tag.insert(0, "tag_id", range(1, len(fact_tag) + 1))
fact_tag["date_id"] = to_date_id(fact_tag["tagged_at"])
fact_tag = fact_tag[["tag_id","user_id","movie_id","date_id","tag","tagged_at"]]

### dim_user – profil uživatele

MovieLens o uživatelích nic osobního neobsahuje (jen anonymní ID) → popis odvodíme z chování:
- `n_ratings` – aktivita, `user_avg_rating` – „přísnost“,
- `activity_cat` – kategorie podle kvartilů aktivity (každá skupina má zhruba čtvrtinu uživatelů),
- `strictness_cat` – přísný / průměrný / benevolentní podle průměru vůči celkovému průměru.

In [15]:
u = fact_rating.groupby("user_id").agg(n_ratings=("rating","size"),
                                        user_avg_rating=("rating","mean"),
                                        first_rating=("rated_at","min"),
                                        last_rating=("rated_at","max")).reset_index()
u["active_days"] = (u["last_rating"] - u["first_rating"]).dt.days
u["activity_cat"] = pd.qcut(u["n_ratings"].rank(method="first"), 4,
                            labels=["nízká","střední","vysoká","velmi vysoká"])
overall = fact_rating["rating"].mean()
u["strictness_cat"] = pd.cut(u["user_avg_rating"] - overall, bins=[-10, -0.25, 0.25, 10],
                             labels=["přísný","průměrný","benevolentní"])
dim_user = u
dim_user.head()

,user_id,n_ratings,user_avg_rating,first_rating,last_rating,active_days,activity_cat,strictness_cat
0,13,412,3.559466,2009-03-25 08:25:39,2011-11-02 20:30:03,952,velmi vysoká,průměrný
1,21,103,4.233010,2016-02-24 03:27:49,2016-02-24 03:46:50,0,vysoká,benevolentní
2,25,87,3.528736,1996-07-01 10:28:14,1996-07-01 10:46:27,0,vysoká,průměrný
3,45,44,3.795455,1996-08-08 21:39:17,1996-08-08 21:52:03,0,střední,průměrný
4,85,68,3.544118,1996-10-05 21:17:34,1996-10-05 21:37:50,0,střední,průměrný


## 10. Ověření správnosti

Zadání výslovně chce **ověřit spojení** – „počty záznamů na zdrojích a ve výsledku“.
Výsledek této tabulky si dejte na slide.

In [16]:
checks = pd.DataFrame([
    ["movies (zdroj) = dim_movie",          src_counts["movies"],  len(dim_movie)],
    ["ratings (vzorek) = fact_rating",      src_counts["ratings_sample"], len(fact_rating)],
    ["tags (vzorek) = fact_tag",            src_counts["tags_sample"],    len(fact_tag)],
    ["fact_tag bez uživatele v dim_user",   0, (~fact_tag["user_id"].isin(dim_user["user_id"])).sum()],
    ["fact_rating bez filmu v dim_movie",   0, (~fact_rating["movie_id"].isin(dim_movie["movie_id"])).sum()],
    ["fact_rating bez uživatele v dim_user",0, (~fact_rating["user_id"].isin(dim_user["user_id"])).sum()],
    ["fact_rating bez data v dim_date",     0, (~fact_rating["date_id"].isin(dim_date["date_id"])).sum()],
    ["fact_tag bez filmu v dim_movie",      0, (~fact_tag["movie_id"].isin(dim_movie["movie_id"])).sum()],
    ["duplicitní movie_id v dim_movie",     0, dim_movie["movie_id"].duplicated().sum()],
    ["duplicitní rating_id",                0, fact_rating["rating_id"].duplicated().sum()],
], columns=["kontrola","očekáváno","skutečnost"])
checks["OK"] = checks["očekáváno"] == checks["skutečnost"]
display(checks)
assert checks["OK"].all(), "Některá kontrola neprošla – podívejte se do tabulky výše."

,kontrola,očekáváno,skutečnost,OK
0,movies (zdroj) = dim_movie,62423,62423,True
1,ratings (vzorek) = fact_rating,2503205,2503205,True
2,tags (vzorek) = fact_tag,77672,77672,True
3,fact_tag bez uživatele v dim_user,0,0,True
4,fact_rating bez filmu v dim_movie,0,0,True
5,fact_rating bez uživatele v dim_user,0,0,True
6,fact_rating bez data v dim_date,0,0,True
7,fact_tag bez filmu v dim_movie,0,0,True
8,duplicitní movie_id v dim_movie,0,0,True
9,duplicitní rating_id,0,0,True


### Splnění požadavku „min. 10 tis. řádků a 30 sloupců spojeného datasetu“

Spojíme fakt hodnocení se všemi dimenzemi do jedné široké tabulky (tzv. denormalizovaný pohled).
Ten slouží jen ke kontrole velikosti – do databáze jdou normalizované tabulky výše.

In [17]:
wide = (fact_rating
        .merge(dim_movie, on="movie_id", how="left", validate="many_to_one")
        .merge(dim_user,  on="user_id",  how="left", validate="many_to_one")
        .merge(dim_date,  on="date_id",  how="left", validate="many_to_one", suffixes=("", "_date")))
print(f"Spojený dataset: {wide.shape[0]:,} řádků × {wide.shape[1]} sloupců")
assert wide.shape[0] >= 10_000 and wide.shape[1] >= 30

Spojený dataset: 2,503,205 řádků × 47 sloupců


## 11. Krátký profil dat (podklad pro prezentaci)

In [18]:
print("Hodnocení – rozložení:")
print(fact_rating["rating"].value_counts().sort_index())
print("\nFilmy podle dekády:")
print(dim_movie["decade"].value_counts().sort_index())
print("\nKorelace průměru MovieLens (×2) a IMDb ratingu:",
      round(dim_movie[["ml_avg_rating_10","imdb_rating"]].corr().iloc[0,1], 3))
print("\nPrůměrný rozdíl IMDb − ML:", round(dim_movie["rating_diff"].mean(), 3))
print("Spolehlivých filmů:", dim_movie["spolehlivy"].sum())

Hodnocení – rozložení:
rating
0.5     36412
1.0     77047
1.5     41184
2.0    162047
2.5    125548
3.0    483836
3.5    319140
4.0    662595
4.5    222616
5.0    372780
Name: count, dtype: int64

Filmy podle dekády:
decade
1870        2
1880        7
1890      107
1900      156
1910      240
1920      574
1930     2073
1940     2329
1950     2978
1960     3602
1970     4975
1980     5218
1990     6708
2000    12752
2010    20683
2020       16
Name: count, dtype: Int64

Korelace průměru MovieLens (×2) a IMDb ratingu: 0.534

Průměrný rozdíl IMDb − ML: 0.082
Spolehlivých filmů: 24231


## 12. Export do `data/processed/`

Každá tabulka = jeden CSV soubor = jedna tabulka v PostgreSQL.
- oddělovač čárka, kódování UTF-8, desetinná tečka,
- prázdná hodnota = prázdné pole (PostgreSQL ji při importu načte jako NULL).

⚠️ `fact_rating.csv` bude mít stovky MB → na GitHub se nevejde (limit 100 MB).
Do repozitáře dejte notebook (data jdou kdykoli znovu vytvořit) a zip s daty případně na OneDrive.

In [19]:
tables = {
    "dim_movie": dim_movie, "dim_genre": dim_genre, "bridge_movie_genre": bridge_movie_genre,
    "dim_person": dim_person, "bridge_movie_director": bridge_movie_director,
    "dim_user": dim_user, "dim_date": dim_date,
    "fact_rating": fact_rating, "fact_tag": fact_tag,
}
summary = []
for name, df in tables.items():
    path = f"{OUT}/{name}.csv"
    df.to_csv(path, index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
    summary.append([name, len(df), df.shape[1], round(os.path.getsize(path)/1e6, 2)])
summary = pd.DataFrame(summary, columns=["tabulka","řádků","sloupců","MB"])
summary.to_csv(f"{OUT}/_row_counts.csv", index=False)   # pro kontrolu po nahrání do DB
summary

,tabulka,řádků,sloupců,MB
0,dim_movie,62423,28,14.68
1,dim_genre,19,2,0.00
2,bridge_movie_genre,107245,2,0.94
3,dim_person,28277,5,1.72
4,bridge_movie_director,70549,2,1.17
5,dim_user,16254,8,1.36
6,dim_date,8405,7,0.36
7,fact_rating,2503205,7,136.23
8,fact_tag,77672,6,4.55


In [20]:
# V Colabu: zabalit do zipu a stáhnout k sobě do počítače
import shutil
shutil.make_archive("processed", "zip", OUT)
try:
    from google.colab import files
    files.download("processed.zip")
except ImportError:
    print("Nejste v Colabu – soubory jsou ve složce", OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>